# Gym Exercise Classification: RecGym Dataset

## Section 0: Context and Overarching Question

Where notebook 03 used the PAMAP2 dataset of daily activities as a proxy for the
lab-to-field generalisation problem, this notebook works directly with the RecGym dataset
(UCI, 2024): a collection of actual gym workout recordings. Subjects performed exercises
including squats, bench press, arm curls, leg press, leg curl, adductor machine, rope
skipping, and cardio machines, while wearing IMU devices simultaneously on the wrist, in a
trouser pocket, and on the leg.

This makes the data substantially more relevant to Fort than PAMAP2. The exercise labels
match Fort's target exercise library directly, the sensor positions correspond to deployment
options Fort is evaluating, and the Null periods (rest between sets) represent the set
detection problem that sits underneath exercise classification in real workout tracking.

**Dataset summary:**
- 10 subjects, 5 sessions each
- 3 sensor positions recorded simultaneously: wrist, pocket, leg
- Sensor channels: 3-axis accelerometer, 3-axis gyroscope, 1 capacitive sensor
- 11 exercise classes plus a Null (rest) class
- ~4.7 million rows total, ~1.6 million per sensor position

**The same product constraints from notebook 03 apply here.** Errors of commission remain
worse than errors of omission. The 0.85 per-class F1 threshold remains the launch gate.
The key new question this dataset answers directly: which of Fort's target exercises are
tractable from the wrist sensor alone, and which require a second sensor at the leg?


In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

import src.preprocessing as pp
import src.features as feat

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

DATA_PATH   = ROOT / "data" / "raw" / "recgym" / "RecGym.csv"
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

# RecGym data was collected at 50 Hz (from dataset documentation).
# No timestamp column is present in the CSV, so the rate is treated as a known constant.
SAMPLE_RATE = 50
WINDOW_SIZE = 100   # 2 seconds at 50 Hz
OVERLAP     = 0.5

SENSOR_COLS = ["A_x", "A_y", "A_z", "G_x", "G_y", "G_z", "C_1"]
IMU_COLS    = ["A_x", "A_y", "A_z", "G_x", "G_y", "G_z"]   # accelerometer + gyroscope only
LABEL_COL   = "Workout"
SUBJECT_COL = "Subject"

F1_THRESHOLD = 0.85

print(f"Project root: {ROOT}")
print("Setup complete.")


---

## Section 1: Gym Exercise Classification

**Core question:** How well does a model trained on some subjects generalise to an unseen
subject, using only the wrist sensor?

The wrist is Fort's primary sensor. Per-class performance on the wrist sensor with real gym
exercise data gives a direct read on which exercises can ship at launch and which require more
data or a second sensor.

**Evaluation standard:** leave-one-subject-out cross-validation across 10 subjects.

---

### 1.1 Load and inspect the dataset


In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape:     {df.shape}")
print(f"Columns:   {list(df.columns)}")
print(f"Subjects:  {sorted(df[SUBJECT_COL].unique())}")
print(f"Positions: {list(df['Position'].unique())}")
print()
print("Class distribution (all positions combined):")
print(df[LABEL_COL].value_counts().to_string())
print()
print("Rows per sensor position:")
print(df["Position"].value_counts().to_string())


### 1.2 Preprocessing and windowing

The pipeline mirrors notebook 03:

1. Filter to the wrist position.
2. Segment 2-second windows (100 samples at 50 Hz) with 50% overlap, grouped by
   (subject, exercise) so no window straddles two different activities.
3. Extract time-domain and frequency-domain features per channel.

The Null class (rest periods between sets) is retained as a label. Fort needs to detect when
a set ends, which requires the model to distinguish active exercise from rest. Null windows
are the training signal for that capability.


In [ ]:
def build_feature_df(df_all, position, sensor_cols=SENSOR_COLS):
    """Filter to one sensor position, segment windows, and extract features."""
    df_pos = df_all[df_all["Position"] == position].reset_index(drop=True)

    X, y, subjects = pp.segment_windows(
        df_pos,
        feature_cols=sensor_cols,
        label_col=LABEL_COL,
        subject_col=SUBJECT_COL,
        window_size=WINDOW_SIZE,
        overlap=OVERLAP,
    )
    feat_df = feat.build_feature_matrix(X, axis_names=sensor_cols, sample_rate=SAMPLE_RATE)
    feat_df["label"]   = y
    feat_df["subject"] = subjects.astype(int)
    return feat_df


WRIST_CACHE = ROOT / "data" / "processed" / "recgym_wrist_features.parquet"

if WRIST_CACHE.exists():
    print(f"Loading cached wrist feature matrix from {WRIST_CACHE}...")
    wrist_df = pd.read_parquet(WRIST_CACHE)
    print("Loaded in seconds.")
else:
    print("Building wrist feature matrix (allow 5-10 minutes)...")
    wrist_df = build_feature_df(df, position="wrist")
    WRIST_CACHE.parent.mkdir(parents=True, exist_ok=True)
    wrist_df.to_parquet(WRIST_CACHE, index=False)
    print(f"Saved to {WRIST_CACHE}")

feature_cols = [c for c in wrist_df.columns if c not in ("label", "subject")]
print(f"Windows:  {wrist_df.shape[0]:,}")
print(f"Features: {len(feature_cols)}")
print()
print("Windows per class (wrist):")
print(wrist_df["label"].value_counts().to_string())


### 1.3 Leave-one-subject-out cross-validation


In [ ]:
def run_loso_cv(df, feature_cols, n_estimators=200):
    """Leave-one-subject-out CV. Returns (y_true, y_pred) pooled across all folds."""
    subj_ids = sorted(df["subject"].unique())
    all_true, all_pred = [], []

    for i, test_subj in enumerate(subj_ids):
        train = df[df["subject"] != test_subj]
        test  = df[df["subject"] == test_subj]

        X_tr, y_tr = train[feature_cols].values, train["label"].values
        X_te, y_te = test[feature_cols].values,  test["label"].values

        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)

        all_true.extend(y_te)
        all_pred.extend(y_pred)

        acc = (y_pred == y_te).mean()
        print(f"  Subject {test_subj} ({i + 1}/{len(subj_ids)}): accuracy = {acc:.3f}")

    return np.array(all_true), np.array(all_pred)


print("Running LOSO cross-validation (200 trees x 10 folds)...")
y_true, y_pred = run_loso_cv(wrist_df, feature_cols)

overall_acc = (y_true == y_pred).mean()
macro_f1    = f1_score(y_true, y_pred, average="macro")
print(f"\nOverall accuracy: {overall_acc:.3f}")
print(f"Macro F1:         {macro_f1:.3f}")


### 1.4 Per-class results


In [ ]:
classes   = sorted(np.unique(y_true))
f1_scores = f1_score(y_true, y_pred, labels=classes, average=None)

order   = np.argsort(f1_scores)[::-1]
s_names = [classes[i] for i in order]
s_f1    = [f1_scores[i] for i in order]

colors = [
    "#27ae60" if v >= F1_THRESHOLD else
    "#e67e22" if v >= 0.70 else
    "#c0392b"
    for v in s_f1
]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(s_names, s_f1, color=colors, edgecolor="white", height=0.65)
ax.axvline(
    F1_THRESHOLD, color="#2c3e50", linewidth=1.5, linestyle="--", zorder=5,
)
ax.set_xlabel("Per-class F1  (LOSO, n = 10 held-out subjects)")
ax.set_title("Gym exercise classification: per-class F1, wrist sensor", fontweight="bold")
ax.set_xlim(0, 1.08)
ax.tick_params(axis="y", labelsize=10)

for bar, val in zip(bars, s_f1):
    ax.text(
        val + 0.012, bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}", va="center", fontsize=9,
    )

patch_ship   = mpatches.Patch(color="#27ae60", label="Ready to ship  (F1 >= 0.85)")
patch_review = mpatches.Patch(color="#e67e22", label="Frustration zone  (0.70 to 0.85)")
patch_hold   = mpatches.Patch(color="#c0392b", label="Not yet shippable  (F1 < 0.70)")
ax.legend(handles=[patch_ship, patch_review, patch_hold], loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "gym_per_class_f1_wrist.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSummary by tier:")
for name, score in zip(s_names, s_f1):
    tier = "SHIP  " if score >= F1_THRESHOLD else ("REVIEW" if score >= 0.70 else "HOLD  ")
    print(f"  [{tier}]  {name:20s}  F1 = {score:.3f}")


In [ ]:
cm      = confusion_matrix(y_true, y_pred, labels=classes)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes,
    ax=ax,
    vmin=0,
    vmax=1,
    linewidths=0.4,
    cbar_kws={"shrink": 0.75},
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True",      fontsize=12)
ax.set_title(
    "Confusion matrix: gym exercises, wrist sensor  (LOSO, normalised by true class)",
    fontweight="bold",
)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0,  fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "gym_confusion_matrix_wrist.png", dpi=150, bbox_inches="tight")
plt.show()


### 1.5 Alternative method: Hidden Markov Models

A Gaussian HMM treats each 2-second window as a time series of observations moving
through a small number of hidden states. Where the Random Forest collapses the window
into a flat feature vector and ignores temporal order, the HMM explicitly models
which state the motion is in at each timestep and the probability of transitioning to
the next state.

For exercise classification this is a natural fit. A bicep curl has a recognisable
phase structure: setup, concentric (lifting), peak, eccentric (lowering). An HMM
trained on curl windows learns to expect those transitions in that order, which may
separate exercises that differ in phase structure rather than raw amplitude.

**Implementation.** One GaussianHMM is trained per exercise class on all training
windows for that class. At inference, the log-likelihood of each test window is
computed under every class model and the highest-scoring class is predicted. Training
is capped at 500 windows per class per fold to keep each fold tractable.
Evaluation uses the same LOSO protocol as the Random Forest for a direct comparison.


In [ ]:
from hmmlearn import hmm as hmmlib


def train_class_hmms(X_windows, y, n_states=4, max_windows=500):
    """
    Train one GaussianHMM per class on raw window sequences.

    X_windows : (n_windows, window_size, n_channels)
    y         : (n_windows,) string labels
    Returns dict mapping class label -> fitted GaussianHMM (or None on failure).
    """
    classes = sorted(np.unique(y))
    models  = {}
    rng     = np.random.default_rng(RANDOM_STATE)

    for cls in classes:
        cls_windows = X_windows[y == cls]

        if len(cls_windows) > max_windows:
            idx = rng.choice(len(cls_windows), max_windows, replace=False)
            cls_windows = cls_windows[idx]

        n_win, win_sz, n_ch = cls_windows.shape
        X_concat = cls_windows.reshape(n_win * win_sz, n_ch)
        lengths  = [win_sz] * n_win

        model = hmmlib.GaussianHMM(
            n_components=n_states,
            covariance_type="diag",
            n_iter=20,
            random_state=RANDOM_STATE,
        )
        try:
            model.fit(X_concat, lengths)
            models[cls] = model
        except Exception:
            models[cls] = None

    return models


def hmm_predict(models, window):
    """Score a single window (window_size, n_channels) under each class HMM."""
    best_cls, best_score = None, -np.inf
    for cls, model in models.items():
        if model is None:
            continue
        try:
            score = model.score(window)
        except Exception:
            score = -np.inf
        if score > best_score:
            best_score = score
            best_cls   = cls
    return best_cls


def run_hmm_loso(X, y, subjects, n_states=4):
    """LOSO CV using per-class GaussianHMMs. Returns (y_true, y_pred)."""
    subj_ids = sorted(np.unique(subjects))
    all_true, all_pred = [], []

    for i, test_subj in enumerate(subj_ids):
        tr = subjects != test_subj
        te = subjects == test_subj

        models = train_class_hmms(X[tr], y[tr], n_states=n_states)
        preds  = [hmm_predict(models, w) for w in X[te]]

        all_true.extend(y[te])
        all_pred.extend(preds)

        acc = np.mean(np.array(preds) == y[te])
        print(f"  Subject {test_subj} ({i + 1}/{len(subj_ids)}): accuracy = {acc:.3f}")

    return np.array(all_true), np.array(all_pred)


# Re-segment wrist data to get raw windows (fast -- no feature computation).
print("Segmenting wrist windows for HMM...")
df_wrist = df[df["Position"] == "wrist"].reset_index(drop=True)
X_raw, y_raw, subj_raw = pp.segment_windows(
    df_wrist,
    feature_cols=SENSOR_COLS,
    label_col=LABEL_COL,
    subject_col=SUBJECT_COL,
    window_size=WINDOW_SIZE,
    overlap=OVERLAP,
)
print(f"Windows: {X_raw.shape[0]:,}  shape per window: {X_raw.shape[1:]}")

print("\nRunning HMM LOSO (4 hidden states, up to 500 training windows per class)...")
y_true_hmm, y_pred_hmm = run_hmm_loso(X_raw, y_raw, subj_raw, n_states=4)

hmm_acc    = (y_true_hmm == y_pred_hmm).mean()
hmm_macro  = f1_score(y_true_hmm, y_pred_hmm, average="macro")
print(f"\nHMM overall accuracy: {hmm_acc:.3f}")
print(f"HMM macro F1:         {hmm_macro:.3f}")


In [ ]:
hmm_classes = sorted(np.unique(y_true_hmm))
hmm_f1_arr  = f1_score(y_true_hmm, y_pred_hmm, labels=hmm_classes, average=None)
hmm_f1      = dict(zip(hmm_classes, hmm_f1_arr))

# RF F1 already computed earlier as f1_scores / classes
rf_f1 = dict(zip(classes, f1_scores))

# Sort by RF F1 descending so the chart reads consistently with Section 1.4
order       = sorted(classes, key=lambda c: rf_f1.get(c, 0), reverse=True)
rf_vals     = [rf_f1.get(c, 0)  for c in order]
hmm_vals    = [hmm_f1.get(c, 0) for c in order]

y_pos = np.arange(len(order))
height = 0.38

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(y_pos + height / 2, rf_vals,  height, label="Random Forest", color="#2980b9", edgecolor="white")
ax.barh(y_pos - height / 2, hmm_vals, height, label="Gaussian HMM",  color="#e67e22", edgecolor="white")

ax.axvline(F1_THRESHOLD, color="#2c3e50", linewidth=1.5, linestyle="--",
           label=f"Launch threshold  (F1 = {F1_THRESHOLD})")
ax.set_yticks(y_pos)
ax.set_yticklabels(order, fontsize=10)
ax.set_xlabel("Per-class F1  (LOSO, n = 10 held-out subjects)")
ax.set_title("Random Forest vs Gaussian HMM: per-class F1, wrist sensor", fontweight="bold")
ax.set_xlim(0, 1.12)
ax.legend(loc="lower right", framealpha=0.9)

for i, (rf_v, hmm_v) in enumerate(zip(rf_vals, hmm_vals)):
    ax.text(rf_v  + 0.01, i + height / 2, f"{rf_v:.2f}",  va="center", fontsize=8, color="#1a5276")
    ax.text(hmm_v + 0.01, i - height / 2, f"{hmm_v:.2f}", va="center", fontsize=8, color="#784212")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "gym_rf_vs_hmm.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPer-class delta (HMM minus RF):")
for cls in order:
    delta = hmm_f1.get(cls, 0) - rf_f1.get(cls, 0)
    direction = "HMM better" if delta > 0.02 else ("RF better" if delta < -0.02 else "similar")
    print(f"  {cls:20s}  {delta:+.3f}  ({direction})")


**What the RF vs HMM comparison reveals.**

The two models make different errors and have different failure modes. Where the RF
sees a flat feature vector, the HMM sees a sequence -- so exercises that share similar
aggregate statistics but differ in temporal phase structure should favour the HMM.
Exercises that are distinguished primarily by amplitude or frequency content should
favour the RF, which has direct features for both.

A practical implication: if the HMM does substantially better on a specific exercise
class, that class has phase structure the feature engineering is not capturing. Adding
temporal features (e.g., autocorrelation at the dominant period, phase-aligned
statistics) to the RF feature set is one way to close that gap without adopting the
generative model for production. A generative model also enables out-of-distribution
detection: if a test window has low log-likelihood under all class HMMs, the model
can abstain rather than guess, which is the error-of-omission behaviour Fort wants.

For production, the RF is the more practical choice: faster to train, faster to
predict, less sensitive to initialisation, and no per-class model management. The HMM
comparison is most useful as a diagnostic -- showing where temporal structure matters
and informing which feature engineering investments have the highest return.


### What the results mean for Fort

**Upper body exercises are the wrist sensor's natural domain.** ArmCurl and BenchPress place
the wrist closest to the primary kinematic source. High F1 for these classes gives Fort a
clear v1 launch foundation for the upper body library without any additional sensors.

**Lower body machine exercises are the hard cases.** Squat, LegPress, LegCurl, and Adductor
are performed with the arms relatively fixed. The wrist sensor captures secondary vibration
and arm stabilisation rather than the primary leg motion. Lower F1 for these classes is
expected and consistent with the PAMAP2 findings in notebook 03. Section 2 quantifies
exactly how much a leg sensor closes this gap.

**Null class performance reflects set detection capability.** A high Null F1 means the model
can reliably separate "exercising" from "resting", which is the set detection sub-problem
Fort needs to solve before rep counting makes sense. This is a distinct capability from
exercise classification. Both need to work for the end-to-end product to function.

**Confusion matrix priorities.** Systematic off-diagonal mass between two exercises represents
errors of commission: the model confidently names the wrong exercise. These pairs are the
first targets for additional training data and UX disambiguation prompts.


---

## Section 2: Sensor Position Comparison

**Core question:** How much does a wrist sensor miss compared to a leg sensor, and for which
specific exercises?

RecGym records the same session simultaneously from wrist, pocket, and leg. This is the most
direct evidence available for the sensor placement question: same subjects, same exercises,
same sessions, different sensor positions. No inference is required about whether the result
generalises to other datasets.

The pocket sensor is a proxy for a phone in a trouser pocket, which is a zero-cost path to
lower body signal. If pocket F1 is close to leg F1 for the target lower body exercises, phone
IMU fusion is a viable alternative to dedicated hardware.

**Approach:** LOSO cross-validation run independently for each sensor position using IMU
channels only (no capacitive sensor), so the comparison is fair across positions.

---

### 2.1 Build feature matrices for leg and pocket positions


In [ ]:
# IMU-only features for a fair cross-position comparison.
# The capacitive sensor measures skin contact, which has different physical meaning at the
# wrist, leg, and pocket -- including it would confound the position comparison.

print("Building leg feature matrix...")
leg_df = build_feature_df(df, position="leg", sensor_cols=IMU_COLS)
print(f"  Windows: {leg_df.shape[0]:,}")

print("\nBuilding pocket feature matrix...")
pocket_df = build_feature_df(df, position="pocket", sensor_cols=IMU_COLS)
print(f"  Windows: {pocket_df.shape[0]:,}")

print("\nRebulding wrist feature matrix (IMU only, for fair comparison)...")
wrist_imu_df = build_feature_df(df, position="wrist", sensor_cols=IMU_COLS)
print(f"  Windows: {wrist_imu_df.shape[0]:,}")

imu_feature_cols = [c for c in wrist_imu_df.columns if c not in ("label", "subject")]
print(f"\nIMU feature columns: {len(imu_feature_cols)}")


In [ ]:
# Run LOSO for each position. 100 estimators to keep total runtime manageable
# across three 10-fold runs.
print("Running LOSO for each sensor position (100 trees x 10 folds x 3 positions)...")

position_results = {}

for label, pos_df in [("Wrist", wrist_imu_df), ("Leg", leg_df), ("Pocket", pocket_df)]:
    print(f"\n  {label}:")
    yt, yp = run_loso_cv(pos_df, imu_feature_cols, n_estimators=100)
    macro  = f1_score(yt, yp, average="macro")
    position_results[label] = (yt, yp)
    print(f"  => Macro F1: {macro:.3f}")


### 2.2 F1 delta by exercise class


In [ ]:
f1_by_pos = {}
for label, (yt, yp) in position_results.items():
    pos_classes = sorted(np.unique(yt))
    f1_by_pos[label] = dict(zip(
        pos_classes,
        f1_score(yt, yp, labels=pos_classes, average=None),
    ))

shared_classes = sorted(set(f1_by_pos["Wrist"]) & set(f1_by_pos["Leg"]))

delta_leg = {
    c: f1_by_pos["Leg"][c] - f1_by_pos["Wrist"][c]
    for c in shared_classes
}
delta_sorted = dict(sorted(delta_leg.items(), key=lambda kv: kv[1], reverse=True))

fig, ax = plt.subplots(figsize=(11, 5))
d_colors = ["#27ae60" if v >= 0 else "#c0392b" for v in delta_sorted.values()]
ax.barh(
    list(delta_sorted.keys()),
    list(delta_sorted.values()),
    color=d_colors,
    edgecolor="white",
    height=0.65,
)
ax.axvline(0, color="#2c3e50", linewidth=1.0)
ax.set_xlabel("F1 delta  (leg sensor vs wrist sensor)")
ax.set_title(
    "Sensor position comparison: per-exercise F1 gain from a leg sensor",
    fontweight="bold",
)
for i, (name, val) in enumerate(delta_sorted.items()):
    offset = 0.008 if val >= 0 else -0.008
    ha     = "left"  if val >= 0 else "right"
    ax.text(val + offset, i, f"{val:+.3f}", va="center", ha=ha, fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "gym_position_delta.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
rows = []
for c in shared_classes:
    wrist_f1  = f1_by_pos["Wrist"].get(c, float("nan"))
    pocket_f1 = f1_by_pos["Pocket"].get(c, float("nan"))
    leg_f1    = f1_by_pos["Leg"].get(c, float("nan"))
    rows.append({
        "Exercise":             c,
        "Wrist F1":             round(wrist_f1,  3),
        "Pocket F1":            round(pocket_f1, 3),
        "Leg F1":               round(leg_f1,    3),
        "Delta (leg vs wrist)": round(leg_f1 - wrist_f1, 3),
    })

summary_df = (
    pd.DataFrame(rows)
    .set_index("Exercise")
    .sort_values("Delta (leg vs wrist)", ascending=False)
)

summary_df.style.format("{:.3f}").background_gradient(
    subset=["Wrist F1", "Pocket F1", "Leg F1"],
    cmap="RdYlGn",
    vmin=0.5,
    vmax=1.0,
)


### What the results mean for Fort

**Lower body exercises show the largest leg sensor gain.** Squat, LegPress, LegCurl, and
Adductor all involve the legs as the primary kinematic source. A sensor at the thigh or shin
measures that motion directly. The F1 delta for these classes is the empirical case for a
second sensor -- not an assumption derived from biomechanics, but a measurement from the same
subjects performing the same exercises in the same session.

**Upper body exercises are stable or negative for the leg sensor.** ArmCurl and BenchPress
show small or negative deltas, confirming the wrist is the correct sensor for these
exercises. A leg sensor adds noise rather than signal for upper body movements.

**The pocket column is the phone IMU proxy.** If pocket F1 is within 0.05 of leg F1 for
target lower body exercises, a phone in a trouser pocket provides a viable lower body signal
at zero hardware cost. This is the go/no-go gate for phone IMU fusion in the roadmap: measure
it, do not assume it. The table above gives that measurement directly.

**Graceful degradation for wrist-only lower body.** For exercises where the wrist sensor F1
is below 0.85 and a leg sensor or phone is not available, the correct response is a
low-confidence prompt rather than a committed classification. An error of omission ("was that
a squat?") preserves trust. An error of commission ("Squat: 8 reps" when it was a leg press)
does not.


---

## Section 3: What Good Looks Like

The numeric targets are unchanged from notebook 03 because they are derived from adoption
constraints, not from the specific dataset. The RecGym results now give Fort direct per-class
F1 evidence on actual gym exercises to validate or challenge each target.

---

### Key targets

| Metric | Target | Basis |
|---|---|---|
| Per-class LOSO F1 | >= 0.85 per exercise | Below this, first-session visible errors are likely |
| Rep count MAE | < 1.5 reps per set | Above this, the count error is salient to most athletes |
| User correction rate | < 15% of sets within 6 months | Above this, the product still feels unfinished |

### What RecGym adds relative to PAMAP2

The PAMAP2 analysis in notebook 03 established the generalisation framework and the sensor
constraint argument. RecGym replaces proxy exercises (Nordic walking, ironing) with the
actual exercises in Fort's target library. The per-class F1 results from Section 1 are
therefore a direct launch checklist: exercises above 0.85 can ship, exercises between 0.70
and 0.85 go behind a low-confidence UX, and exercises below 0.70 are data collection targets.

The position comparison in Section 2 provides the sensor hardware decision in a form that can
be taken directly to a product or investor conversation: here are the exercises that improve
by more than 0.05 F1 with a leg sensor, and here is whether the phone pocket closes most of
that gap.

### Competitive context

Apple Watch identifies exercise type at 60 to 75% accuracy across its supported activity list
(Khushhal et al., 2017). Apple's strategy is explicit scope control: a narrow, high-confidence
list rather than broad coverage. The Atlas Wristband positioned itself as broad-coverage
strength tracking and failed, with exercise misclassification cited as the primary churn
driver. RecGym gives Fort the evidence to choose scope deliberately rather than reactively:
ship the exercises the data supports, hold the ones it does not.


---

## Section 4: Proposed Solutions and Engineering Roadmap

The roadmap structure mirrors notebook 03 but is updated to reflect what the RecGym results
reveal specifically about gym exercise classification and sensor placement.

---

### Near-term (0 to 3 months)

**What gets built:**
- Lab data collection targeting Fort's exercise library. Use a simultaneous wrist and leg
  recording protocol from day one, even if the leg sensor is not in the v1 product. This data
  costs nothing extra to collect and directly answers the sensor expansion question when needed.
- Baseline classifier using the pipeline in this notebook: 2-second windows at 50 Hz, IMU
  plus capacitive features, random forest with LOSO evaluation by subject.
- Prioritised data collection backlog: the 3 to 5 exercises with the lowest per-class LOSO F1
  from Section 1 are the first collection targets. Additional subjects have the highest
  marginal value here.
- Haptic set confirmation UX from day one: the device signals set completion, shows the
  predicted exercise and rep count, and waits for user confirmation or correction.

**Output:** Working classifier, honest LOSO evaluation report, prioritised data backlog.

**Gate:** All shipped exercises at or above 0.85 per-class LOSO F1. At least 50 labelled
sets per exercise class collected in real gym conditions.

---

### Medium-term (3 to 9 months)

**What gets built:**
- Synthetic augmentation for domain shift robustness: time warping, amplitude scaling,
  Gaussian noise, and sensor axis rotation to simulate different wrist positions and
  inter-subject variation.
- In-app correction pipeline: user corrections stored with raw IMU signal, used to retrain
  the model on a rolling basis.
- Personalised fine-tuning once 20 or more corrected sets per user have accumulated.
- Phone IMU fusion evaluation, with a go/no-go gate driven directly by the Section 2 pocket
  column: if pocket F1 is within 0.05 of leg F1 for the target lower body exercises, phone
  fusion enters the shipping pipeline. If not, dedicated leg hardware is scoped.
- Capacitive sensor investigation: the C_1 channel is underexplored in this analysis. If
  capacitive skin contact at the wrist correlates with grip changes between exercise types
  (barbell versus machine handle versus bodyweight), it may separate exercises that are
  ambiguous from the IMU alone.

**Output:** Improving model, functioning correction UX, resolved decision on phone fusion
and capacitive signal utility.

**Gate:** User correction rate trending below 15% of sets. Personalised models demonstrably
outperforming the global model for users with 20 or more corrected sets.

---

### Long-term (9 to 24 months)

**What gets built:**
- Continuous retraining pipeline with automated LOSO validation before any model deployment.
  Regressions on any shipped exercise class block deployment automatically.
- Federated learning for privacy-preserving personalisation: gradient updates aggregated
  server-side, no raw IMU data centralised.
- Optional satellite sensors: a leg pod or ankle tracker for users who want full lower body
  tracking. The wrist remains the primary device. Lower body exercises unlock when a satellite
  sensor is paired rather than degrading the wrist-only experience.

**Output:** Self-improving system where accuracy increases with field usage.

**Leading indicator:** Aggregate macro F1 on the rolling field test set trending upward
quarter over quarter, with no individual exercise class dropping below 0.85 for more than
one consecutive quarter.
